# Hillsboro GIS Data Acquisition

## Purpose

This notebook acquires immutable raw snapshots of selected City of Hillsboro GIS datasets for the Web Hosted Portfolio project.

## Raw Data Policy

No attribute or geometry transformations are performed during acquisition.

## Pagination

ArcGIS REST services have dataset-specific transfer limits. The acquisition function retrieves records in batches until the complete layer has been downloaded.

## Versioning

New acquisitions receive a new timestamped snapshot directory rather than overwriting previous snapshots.

## Storage

Large raw JSON files are stored outside GitHub in the project's permanent raw-data storage. GitHub contains project code, documentation, and metadata/manifests.

## Layer Metadata (Provenance)

Each ArcGIS REST layer publishes a layer definition (name, id, geometry type, spatial reference, fields, and coded-value domains) in addition to its features. This definition is captured as part of the dated snapshot alongside the raw feature data, using the ArcGIS REST JSON representation (`?f=json`) rather than the human-readable HTML page.

This metadata is provenance for the historical snapshot: it records how coded attribute values (e.g. `PROJECT_TYPE="DEVRE"`) map to their human-readable meanings *at the time of acquisition*. Raw feature data is never decoded or rewritten during acquisition, so this metadata is what allows a snapshot to remain interpretable later even if the live ArcGIS service changes or removes its domains. Cleaning notebooks should read the saved metadata file rather than querying the live service.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import requests

# --------------------------------------------------
# HIL Raw Data Acquisition
# --------------------------------------------------

SNAPSHOT_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

RAW_ROOT = Path(".")

print(f"Snapshot date: {SNAPSHOT_DATE}")
print(f"Raw data root: {RAW_ROOT}")

Snapshot date: 2026-09-15
Raw data root: .


In [2]:
DATASETS = {
    "HIL-001": {
        "name": "semiconductor_businesses",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "EcDev_SemiconductorBusinesses/FeatureServer/0"
        ),
        "source_organization": "City of Hillsboro"
    },

    "HIL-002": {
        "name": "project_boundaries",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "ContructionProjects/FeatureServer/0"
        ),
    },

    "HIL-003": {
        "name": "zoning",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "LandUseGallery_Prod/FeatureServer/23"
        ),
    },

    "HIL-004": {
        "name": "comprehensive_plan",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "LandUseGallery_Prod/FeatureServer/21"
        ),
    },

    "HIL-005": {
        "name": "buildings",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "Planning_BaseData/MapServer/91"
        ),
    },

    "HIL-006": {
        "name": "metro_buildings",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "Planning_BaseData/MapServer/92"
        ),
    },

    "HIL-007": {
        "name": "pavement_projects",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "ContructionProjects/FeatureServer/1"
        ),
    },

    "HIL-008": {
        "name": "city_limits",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "Hillsboro_City_Limits/FeatureServer/0"
        ),
    },

    "HIL-009": {
        "name": "roadway",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "Planning_BaseData/MapServer/80"
        ),
    },
}

print(f"Datasets configured: {len(DATASETS)}")

Datasets configured: 9


In [3]:
def summarize_domains(layer_info):
    """
    Summarize coded-value domains present in a layer definition's fields.

    Returns a list of {field_name, domain_name, coded_values} for every
    field whose domain is a codedValue domain.
    """

    domains = []

    for field in layer_info.get("fields", []) or []:
        domain = field.get("domain")

        if not domain or domain.get("type") != "codedValue":
            continue

        domains.append({
            "field_name": field.get("name"),
            "domain_name": domain.get("name"),
            "coded_values": domain.get("codedValues", []),
        })

    return domains


def acquire_dataset(dataset_id, config):
    """
    Download a complete ArcGIS REST layer as raw JSON.

    - Retrieves records in batches.
    - Preserves source attributes and geometry.
    - Preserves the ArcGIS layer definition (fields, domains, spatial
      reference, geometry type) as provenance metadata.
    - Creates a timestamped snapshot directory.
    - Creates a manifest with metadata and SHA-256 checksum.
    """

    layer_url = config["url"].rstrip("/")
    query_url = f"{layer_url}/query"

    acquired_at = datetime.now(timezone.utc).isoformat()

    # --------------------------------------------------
    # Layer metadata
    # --------------------------------------------------

    layer_response = requests.get(
        layer_url,
        params={"f": "json"},
        timeout=60
    )

    layer_response.raise_for_status()

    layer_info = layer_response.json()

    if "error" in layer_info:
        raise RuntimeError(
            f"Layer metadata error for {dataset_id}: "
            f"{layer_info['error']}"
        )

    max_record_count = layer_info.get(
        "maxRecordCount",
        1000
    )

    coded_domains = summarize_domains(layer_info)

    print(f"Downloading {dataset_id}...")
    print(f"Query URL: {query_url}")
    print(f"Batch size: {max_record_count}")
    print(
        f"Layer metadata: {len(layer_info.get('fields', []) or [])} fields, "
        f"{len(coded_domains)} coded-value domain(s)"
    )

    # --------------------------------------------------
    # Retrieve all records
    # --------------------------------------------------

    all_features = []
    offset = 0
    first_batch = None

    while True:

        params = {
            "where": "1=1",
            "outFields": "*",
            "returnGeometry": "true",
            "resultOffset": offset,
            "resultRecordCount": max_record_count,
            "f": "json"
        }

        response = requests.get(
            query_url,
            params=params,
            timeout=120
        )

        response.raise_for_status()

        batch = response.json()

        if first_batch is None:
            first_batch = batch

        if "error" in batch:
            raise RuntimeError(
                f"ArcGIS error for {dataset_id}: "
                f"{batch['error']}"
            )

        features = batch.get("features", [])

        if not features:
            break

        all_features.extend(features)

        print(
            f"  Retrieved {len(all_features):,} records..."
        )

        if not batch.get(
            "exceededTransferLimit",
            False
        ):
            break

        offset += len(features)

    # --------------------------------------------------
    # Validate acquisition
    # --------------------------------------------------

    record_count = len(all_features)

    if record_count == 0:
        raise RuntimeError(
            f"{dataset_id} returned zero records. "
            "No snapshot was saved."
        )

    # --------------------------------------------------
    # Reconstruct complete response
    # --------------------------------------------------

    data = {
        key: value
        for key, value in first_batch.items()
        if key != "features"
    }

    data["features"] = all_features

    # --------------------------------------------------
    # Create snapshot directories
    # --------------------------------------------------

    snapshot_dir = (
        RAW_ROOT /
        SNAPSHOT_DATE
    )

    datasets_dir = (
        snapshot_dir /
        "datasets" /
        "raw"
    )

    manifests_dir = (
        snapshot_dir /
        "manifests"
    )

    datasets_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    manifests_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------
    # Save raw JSON
    # --------------------------------------------------

    data_file = (
        datasets_dir /
        f"{dataset_id}.json"
    )

    with open(
        data_file,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(data, f)

    # --------------------------------------------------
    # SHA-256 checksum
    # --------------------------------------------------

    sha256 = hashlib.sha256(
        data_file.read_bytes()
    ).hexdigest()

    # --------------------------------------------------
    # Save layer metadata (ArcGIS layer definition)
    # --------------------------------------------------
    # Persisted as-is from the REST layer definition so the snapshot
    # remains interpretable even if the live service later changes.

    metadata_file = (
        datasets_dir /
        f"{dataset_id}_metadata.json"
    )

    with open(
        metadata_file,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            layer_info,
            f,
            indent=2
        )
        f.write("\n")

    metadata_sha256 = hashlib.sha256(
        metadata_file.read_bytes()
    ).hexdigest()

    # --------------------------------------------------
    # Manifest
    # --------------------------------------------------

    manifest = {
        "dataset_id": dataset_id,
        "dataset_name": config["name"],
        "source_organization": (
            config["source_organization"]
        ),
        "source_url": layer_url,
        "query_url": query_url,
        "acquired_at_utc": acquired_at,
        "snapshot_date": SNAPSHOT_DATE,
        "record_count": record_count,
        "file": data_file.name,
        "file_size_bytes": data_file.stat().st_size,
        "sha256": sha256,
        "metadata": {
            "file": metadata_file.name,
            "file_size_bytes": metadata_file.stat().st_size,
            "sha256": metadata_sha256,
            "layer_name": layer_info.get("name"),
            "layer_id": layer_info.get("id"),
            "geometry_type": layer_info.get("geometryType"),
            "spatial_reference": layer_info.get("spatialReference"),
            "field_count": len(layer_info.get("fields", []) or []),
            "coded_value_domain_count": len(coded_domains),
            "coded_value_domain_fields": [
                domain["field_name"] for domain in coded_domains
            ],
        },
        "notes": (
            "Raw acquisition snapshot. "
            "No attribute or geometry "
            "transformations performed. "
            "Layer metadata captured separately "
            "for provenance; coded values in the "
            "raw feature data are left unmodified."
        )
    }

    manifest_file = (
        manifests_dir /
        f"{dataset_id}_manifest.json"
    )

    with open(
        manifest_file,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            manifest,
            f,
            indent=2
        )
        f.write("\n")

    print("\n✓ Complete acquisition")
    print(f"✓ Records: {record_count:,}")
    print(f"✓ Saved: {data_file}")
    print(f"✓ Metadata: {metadata_file}")
    if coded_domains:
        print(
            f"✓ Coded-value domains: {len(coded_domains)} "
            f"({', '.join(manifest['metadata']['coded_value_domain_fields'])})"
        )
    else:
        print("✓ Coded-value domains: none found")
    print(f"✓ Manifest: {manifest_file}")

    return manifest

In [4]:
# Used when downloading a single dataset
manifest = acquire_dataset(
    "HIL-001",
    DATASETS["HIL-001"]
)

Query URL: https://gis.hillsboro-oregon.gov/public/rest/services/public/EcDev_SemiconductorBusinesses/FeatureServer/0/query
Batch size: 2000
Layer metadata: 25 fields, 10 coded-value domain(s)
  Retrieved 59 records...

✓ Complete acquisition
✓ Records: 59
✓ Saved: 2026-09-15\datasets\raw\HIL-001.json
✓ Metadata: 2026-09-15\datasets\raw\HIL-001_metadata.json
✓ Coded-value domains: 10 (INDUSTRY, SUB_INDUSTRY, FAC_TYPE_BACK_OFFICE, FAC_TYPE_DISTRIBUTION, FAC_TYPE_HQ, FAC_TYPE_MANUFACTURING, FAC_TYPE_OFFICE, FAC_TYPE_RD, FAC_TYPE_SALES_SERVICE, SUPPLY_CHAIN)
✓ Manifest: 2026-09-15\manifests\HIL-001_manifest.json


In [5]:
datasets_to_acquire = [
     "HIL-001",
     "HIL-002",
     "HIL-003",
     "HIL-004",
     "HIL-005",
     "HIL-006",
     "HIL-007",
     "HIL-008",
     "HIL-009",
 ]

for dataset_id in datasets_to_acquire:
     print("\n" + "=" * 70)
     acquire_dataset(
         dataset_id,
         DATASETS[dataset_id]
     )


Query URL: https://gis.hillsboro-oregon.gov/public/rest/services/public/EcDev_SemiconductorBusinesses/FeatureServer/0/query
Batch size: 2000
Layer metadata: 25 fields, 10 coded-value domain(s)
  Retrieved 59 records...

✓ Complete acquisition
✓ Records: 59
✓ Saved: 2026-09-15\datasets\raw\HIL-001.json
✓ Metadata: 2026-09-15\datasets\raw\HIL-001_metadata.json
✓ Coded-value domains: 10 (INDUSTRY, SUB_INDUSTRY, FAC_TYPE_BACK_OFFICE, FAC_TYPE_DISTRIBUTION, FAC_TYPE_HQ, FAC_TYPE_MANUFACTURING, FAC_TYPE_OFFICE, FAC_TYPE_RD, FAC_TYPE_SALES_SERVICE, SUPPLY_CHAIN)
✓ Manifest: 2026-09-15\manifests\HIL-001_manifest.json

Query URL: https://gis.hillsboro-oregon.gov/public/rest/services/public/ContructionProjects/FeatureServer/0/query
Batch size: 2000
Layer metadata: 42 fields, 8 coded-value domain(s)
  Retrieved 145 records...

✓ Complete acquisition
✓ Records: 145
✓ Saved: 2026-09-15\datasets\raw\HIL-002.json
✓ Metadata: 2026-09-15\datasets\raw\HIL-002_metadata.json
✓ Coded-value domains: 8 (AGE

In [6]:
# ============================================================
# HIL RAW DATA VALIDATION
# ============================================================

from pathlib import Path
import json
import hashlib

DATA_ROOT = Path(".")
SNAPSHOT_DIR = DATA_ROOT / SNAPSHOT_DATE
DATASETS_DIR = SNAPSHOT_DIR / "datasets" / "raw"
MANIFESTS_DIR = SNAPSHOT_DIR / "manifests"

print("HIL RAW DATA VALIDATION")
print("=" * 70)
print(f"Snapshot: {SNAPSHOT_DATE}")
print(f"Datasets: {DATASETS_DIR}")
print(f"Manifests: {MANIFESTS_DIR}")
print()

dataset_ids = [
    "HIL-001",
    "HIL-002",
    "HIL-003",
    "HIL-004",
    "HIL-005",
    "HIL-006",
    "HIL-007",
    "HIL-008",
    "HIL-009",
]

for dataset_id in dataset_ids:

    data_file = DATASETS_DIR / f"{dataset_id}.json"
    manifest_file = MANIFESTS_DIR / f"{dataset_id}_manifest.json"

    print(f"{dataset_id}")

    # --------------------------------------------------------
    # JSON
    # --------------------------------------------------------

    if not data_file.exists():
        print("  ✗ JSON: missing")
        continue

    size = data_file.stat().st_size

    with open(data_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    records = len(data.get("features", []))

    # --------------------------------------------------------
    # SHA-256
    # --------------------------------------------------------

    sha256 = hashlib.sha256(
        data_file.read_bytes()
    ).hexdigest()

    print(f"  ✓ JSON: {size:,} bytes")
    print(f"  ✓ Records: {records:,}")
    print(f"  ✓ SHA-256: {sha256[:16]}...")

    # --------------------------------------------------------
    # Layer metadata
    # --------------------------------------------------------

    metadata_file = DATASETS_DIR / f"{dataset_id}_metadata.json"

    if not metadata_file.exists():
        print("  ✗ Metadata: missing")
    else:
        with open(metadata_file, "r", encoding="utf-8") as f:
            layer_info = json.load(f)

        field_count = len(layer_info.get("fields", []) or [])
        domain_count = sum(
            1
            for field in layer_info.get("fields", []) or []
            if (field.get("domain") or {}).get("type") == "codedValue"
        )

        print(f"  ✓ Metadata: {metadata_file.stat().st_size:,} bytes")
        print(f"  ✓ Metadata fields: {field_count}")
        print(f"  ✓ Coded-value domains: {domain_count}")

    # --------------------------------------------------------
    # Manifest
    # --------------------------------------------------------

    if manifest_file.exists():
        print("  ✓ Manifest: present")
    else:
        print("  ✗ Manifest: missing")

    print()

print("=" * 70)
print("Validation complete.")

HIL RAW DATA VALIDATION
Snapshot: 2026-09-15
Datasets: 2026-09-15\datasets\raw
Manifests: 2026-09-15\manifests

HIL-001
  ✓ JSON: 52,556 bytes
  ✓ Records: 59
  ✓ SHA-256: a97731f7930c7c74...
  ✓ Metadata: 38,394 bytes
  ✓ Metadata fields: 25
  ✓ Coded-value domains: 10
  ✓ Manifest: present

HIL-002
  ✓ JSON: 693,313 bytes
  ✓ Records: 145
  ✓ SHA-256: 74fd998e026b1b06...
  ✓ Metadata: 51,009 bytes
  ✓ Metadata fields: 42
  ✓ Coded-value domains: 8
  ✓ Manifest: present

HIL-003
  ✓ JSON: 1,433,918 bytes
  ✓ Records: 353
  ✓ SHA-256: 5135b28e0db157c0...
  ✓ Metadata: 65,765 bytes
  ✓ Metadata fields: 9
  ✓ Coded-value domains: 1
  ✓ Manifest: present

HIL-004
  ✓ JSON: 2,383,185 bytes
  ✓ Records: 297
  ✓ SHA-256: a3f3d0a661df2959...
  ✓ Metadata: 31,160 bytes
  ✓ Metadata fields: 9
  ✓ Coded-value domains: 1
  ✓ Manifest: present

HIL-005
  ✓ JSON: 53,686,098 bytes
  ✓ Records: 43,774
  ✓ SHA-256: 3c8a0debd2e5cdcc...
  ✓ Metadata: 23,154 bytes
  ✓ Metadata fields: 27
  ✓ Coded-value 